In [24]:
from math import log

In [25]:
def create_dataset():
    #年龄0-青年 1-中年 2-老年,
    #有工作0-1,
    #有自己的房子0-1,
    #信贷情况0-一般,1-好,2-非常好,
    #判断是否放贷
    data_set = [
        [0, 0, 0, 0, 'no'],         #数据集
        [0, 0, 0, 1, 'no'],
        [0, 1, 0, 1, 'yes'],
        [0, 1, 1, 0, 'yes'],
        [0, 0, 0, 0, 'no'],
        [1, 0, 0, 0, 'no'],
        [1, 0, 0, 1, 'no'],
        [1, 1, 1, 1, 'yes'],
        [1, 0, 1, 2, 'yes'],
        [1, 0, 1, 2, 'yes'],
        [2, 0, 1, 2, 'yes'],
        [2, 0, 1, 1, 'yes'],
        [2, 1, 0, 1, 'yes'],
        [2, 1, 0, 2, 'yes'],
        [2, 0, 0, 0, 'no']
    ]
    labels = ['放贷','不放贷']
    return data_set,labels

### 构建决策树的方法
1. 特征选择
2. 决策树生成
3. 决策树修剪
### 特征选择
选定不同的特征可能会存在不同的结果,需要根据信息量的大小来选择节点构建树
#### 香农熵
信息的度量方式成为熵,该度量方式从前人经验得出
香农熵的计算公式为:
$$H(X)=-\sum_{i=1}^{n}p(x_i)\log_2p(x_i)$$
> n为分类的数目,熵越大,随机变量的不确定性越大
>
根据数据估计得到的熵,为经验熵,经验熵公式为
$$ H(D) = -\sum_{k=1}^{K} \frac{|D_k|}{|D|} \log_2 \frac{|D_k|}{|D|} $$
> D表示样本容量,Dk表示有k个类,每个类的样本容量为|Dk|
>


In [26]:
def calc_shannon_ent(data_set):
    data_set_size = len(data_set)
    label_count = {}
    #对是否放贷进行统计
    for feat_vector in data_set:
        current_label = feat_vector[-1]
        if current_label not in label_count.keys():
            label_count[current_label] = 0
        label_count[current_label] += 1
    shannon_ent = 0.0
    for kv in label_count:
        prob = float(label_count[kv]) / data_set_size
        shannon_ent -= prob * log(prob,2)
    return shannon_ent

### 信息增益
构建决策树的核心就是找到信息增益最大的变量作为根
信息增益越大,对最终分类影响越大,因此还需要求得信息增益
信息增益需要通过条件熵求得
条件熵计算公式
$$ H(Y|X) = \sum_{x \in X} p(x) * H(Y|X=X_i) $$
$$ H(Y|X) = -\sum_{x \in X} p(x) \sum_{y \in Y} p(y|x) \log_2 p(y|x) $$
同理,当熵的概率由数据估计得到的时候,所对应的条件熵为**条件经验熵**

信息增益中,是相对的概念,比如特征A对训练数据集D的信息增益为集合D的经验熵与特征A条件下的D的经验条件熵的差,因此信息增益的计算公式为:
$$ G(D,A) = H(D) - H(D|A) $$
> 一般的,将熵和条件熵作差为互信息,在决策树中,互信息作为信息增益的度量
>
信息增益越大,越适合作为根节点,因为对最终分类效果影响最大

In [27]:
def split_data_set(data_set, idx, value):
    res_data_set = []
    for feat_vec in data_set:
        if feat_vec[idx] == value:
            reduce_feat_vec = feat_vec[:idx]
            reduce_feat_vec.extend(feat_vec[idx+1:])
            res_data_set.append(reduce_feat_vec)
    return res_data_set

def choose_best_feature_to_split(data_set):
    num_feature = len(data_set[0]) - 1
    base_entropy = calc_shannon_ent(data_set)
    best_info_increase = 0.0
    best_feature_idx = -1
    for i in range(num_feature):
        #从data_set取出某一列
        feature_list = [example[i] for example in data_set]
        #使用set去重
        unique_val = set(feature_list)
        condition_entropy = 0.0
        for val in unique_val:
            sub_data_set = split_data_set(data_set, i ,val)
            #计算该分类出现的概率
            prob = len(sub_data_set) / float(len(data_set))
            condition_entropy += prob * calc_shannon_ent(sub_data_set)
        info_increase = base_entropy - condition_entropy
        print("第%d个特征,信息增益为%.3f" % (i, info_increase))
        if info_increase > best_info_increase:
            best_info_increase = info_increase
            best_feature_idx = i
    return best_feature_idx

In [28]:
if __name__ == "__main__":
    data_set, labels = create_dataset()
    best_feature_idx = choose_best_feature_to_split(data_set)
    print("最优特征索引为%d" % best_feature_idx)

第0个特征,信息增益为0.083
第1个特征,信息增益为0.324
第2个特征,信息增益为0.420
第3个特征,信息增益为0.363
最优特征索引为2
